# Performance analysis (public daily figures)

This notebook rebuilds the headline statistics on [jakeliu.me](https://jakeliu.me) from the public file `data/daily.json` only: daily net P&L, drawdown on daily closes, winning-day and winning-week rates, and the longest losing streak.

It does **not** contain strategy code, signals, or per-strategy ledgers.

**Authorship.** I specified the questions, the definitions (America/New_York calendar days, Monday–Sunday weeks, peak-to-trough drawdown on the daily cumulative series), and the checks. An AI assistant wrote the code. I reviewed the outputs against the independently computed ledger totals.

**Window.** 25 September 2026 through 27 November 2026, ending 19:16 America/New_York on 27 November. Figures are settled live P&L after fees, aggregated across strategies, primarily on Kalshi. One duplicate settlement row ($0.19 on 24 November) was removed before this daily file was built.

The official net P&L, **+$17,278.80**, is the sum of settlement-ledger rows. Adding the already-rounded daily figures in this file can differ from that total by a cent.

In [1]:
from pathlib import Path
import json
import sqlite3
import pandas as pd

# Load the public daily file. When this notebook sits in the site repo,
# the data file is at data/daily.json. A local package copy is the fallback.
candidates = [
    Path('data/daily.json'),
    Path('/workspace/career/fix-package/assets/daily.json'),
]
path = next(p for p in candidates if p.exists())
raw = json.loads(path.read_text())
df = pd.DataFrame(raw)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
print('source:', path)
print('days:', len(df), 'from', df.date.min().date(), 'to', df.date.max().date())
df.head()

source: /workspace/career/fix-package/assets/daily.json
days: 64 from 2026-09-25 to 2026-11-27


,date,net_pnl_usd,settled_trades
0,2026-09-25,2.69,70
1,2026-09-26,1.83,113
2,2026-09-27,4.55,103
3,2026-09-28,5.19,111
4,2026-09-29,3.77,182


## SQL over a SQLite table

The same daily rows are loaded into SQLite so the headline statistics are queried, not only computed in pandas.

In [2]:
conn = sqlite3.connect(':memory:')
df.to_sql('daily', conn, index=False, if_exists='replace')

sql = '''
SELECT
  ROUND(SUM(net_pnl_usd), 2) AS net_pnl_usd,
  SUM(settled_trades)        AS settled_trades,
  COUNT(*)                   AS days,
  SUM(CASE WHEN net_pnl_usd > 0 THEN 1 ELSE 0 END) AS profitable_days,
  SUM(CASE WHEN net_pnl_usd < 0 THEN 1 ELSE 0 END) AS losing_days,
  ROUND(AVG(net_pnl_usd), 2) AS avg_day,
  ROUND(MAX(net_pnl_usd), 2) AS best_day,
  ROUND(MIN(net_pnl_usd), 2) AS worst_day
FROM daily
'''
headline = pd.read_sql(sql, conn)
best_date = pd.read_sql(
    'SELECT date FROM daily WHERE net_pnl_usd = (SELECT MAX(net_pnl_usd) FROM daily)',
    conn,
).iloc[0, 0]
worst_date = pd.read_sql(
    'SELECT date FROM daily WHERE net_pnl_usd = (SELECT MIN(net_pnl_usd) FROM daily)',
    conn,
).iloc[0, 0]
headline['best_day_date'] = str(best_date)[:10]
headline['worst_day_date'] = str(worst_date)[:10]
headline

,net_pnl_usd,settled_trades,days,profitable_days,losing_days,avg_day,best_day,worst_day,best_day_date,worst_day_date
0,17278.81,29629,64,55,9,269.98,3377.34,-1447.51,2026-11-26,2026-11-24


## Drawdown on daily closes, streaks, and weekly hit rate

In [3]:
df['cumulative'] = df['net_pnl_usd'].cumsum()
df['peak'] = df['cumulative'].cummax()
df['drawdown'] = df['peak'] - df['cumulative']
daily_close_dd = float(df['drawdown'].max())

streak = 0
longest = 0
start = None
longest_span = None
for date, pnl in zip(df['date'], df['net_pnl_usd']):
    if pnl < 0:
        if streak == 0:
            start = date
        streak += 1
        if streak > longest:
            longest = streak
            longest_span = (start, date)
    else:
        streak = 0

weeks = (
    df.assign(week_start=df['date'] - pd.to_timedelta(df['date'].dt.dayofweek, unit='D'))
      .groupby('week_start', as_index=False)
      .agg(
          net_pnl_usd=('net_pnl_usd', 'sum'),
          settled_trades=('settled_trades', 'sum'),
          days=('date', 'size'),
          profitable_days=('net_pnl_usd', lambda s: int((s > 0).sum())),
      )
)
pos_weeks = int((weeks['net_pnl_usd'] > 0).sum())

print(f'Drawdown on daily closes: ${daily_close_dd:,.2f}')
print(f'Longest losing streak: {longest} day(s) ({longest_span[0].date()} – {longest_span[1].date()})')
print(f'Profitable weeks: {pos_weeks} of {len(weeks)} (Monday–Sunday)')
print(f'Week hit rate: {pos_weeks / len(weeks):.0%}')
print()
print('Official ledger net P&L (not a sum of rounded days): +$17,278.80')
print('Trade-level max drawdown (from settlement rows, not this daily file): $2,462.07')
weeks

Drawdown on daily closes: $1,447.51
Longest losing streak: 2 day(s) (2026-10-14 – 2026-10-15)
Profitable weeks: 10 of 10 (Monday–Sunday)
Week hit rate: 100%

Official ledger net P&L (not a sum of rounded days): +$17,278.80
Trade-level max drawdown (from settlement rows, not this daily file): $2,462.07


/tmp/ipykernel_149766/1731965694.py:22: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df.assign(week_start=df['date'] - pd.to_timedelta(df['date'].dt.dayofweek, unit='D'))


,week_start,net_pnl_usd,settled_trades,days,profitable_days
0,2026-09-21,9.07,286,3,3
1,2026-09-28,120.97,2265,7,7
2,2026-10-05,412.89,2615,7,6
3,2026-10-12,464.00,2465,7,5
4,2026-10-19,518.57,2089,7,5
5,2026-10-26,911.80,1858,7,6
6,2026-11-02,927.64,5223,7,5
7,2026-11-09,3511.91,6443,7,7
8,2026-11-16,4968.60,4287,7,7
9,2026-11-23,5433.36,2098,5,4


## Cross-check

These figures should match the site's full-window record (after the 24 November cleanup): 29,629 settled trades, 55 of 64 days profitable, best day +$3,377.34 on 26 November, worst day −$1,447.51 on 24 November, daily-close drawdown $1,447.51, 10 of 10 weeks profitable, longest losing streak 2 days (14–15 October).

In [4]:
assert int(headline.loc[0, 'settled_trades']) == 29629
assert int(headline.loc[0, 'days']) == 64
assert int(headline.loc[0, 'profitable_days']) == 55
assert abs(float(headline.loc[0, 'best_day']) - 3377.34) < 0.005
assert abs(float(headline.loc[0, 'worst_day']) + 1447.51) < 0.005
assert abs(daily_close_dd - 1447.51) < 0.005
assert longest == 2
assert pos_weeks == 10 and len(weeks) == 10
print('All cross-checks passed.')

All cross-checks passed.
